# Addestramento di Pm projexa (Projexa)

Questo notebook addestra **Pm projexa** (qwen2.5:7b) sulle domande che hai segnato come corrette nella pagina **Analisi PM** di Projexa.
Il risultato è un piccolo file **adattatore** (`pm-projexa-adapter.gguf`, 100-300 MB) da caricare nella stessa pagina (sezione *Modello sulla VM*).

**Prima di iniziare**
1. Menu **Runtime → Cambia tipo di runtime → GPU T4** (gratuita) → Salva.
2. Tieni a portata il file `pm_projexa_addestramento_AAAA-MM-GG.jsonl` scaricato da Analisi PM.

**Poi** menu **Runtime → Esegui tutto**. Alla cella 2 si apre il pulsante per caricare il file .jsonl.
Tempo indicativo: 20-60 minuti in tutto (dipende da quanti esempi). Alla fine il file .gguf si scarica da solo.

Se una cella dà errore, copia il messaggio e mandalo a chi ti segue lo sviluppo: le librerie cambiano spesso versione.

## 1. Installazione delle librerie (5-10 minuti)

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), 'Nessuna GPU: Runtime > Cambia tipo di runtime > GPU T4'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Carica il file degli esempi (.jsonl scaricato da Projexa)

In [ ]:
from google.colab import files
import json
caricati = files.upload()
nome = next(n for n in caricati if n.endswith('.jsonl'))
esempi = [json.loads(r) for r in open(nome, encoding='utf-8') if r.strip()]
utili = sum(1 for e in esempi if e['messages'][-1]['content'] != 'Nessun suggerimento.')
print(f'{len(esempi)} esempi: {utili} domande, {len(esempi) - utili} «Nessun suggerimento»')
if utili < 50:
    print('ATTENZIONE: meno di 50 domande corrette, il modello imparerà poco. Meglio aspettare di averne di più.')

## 3. Modello base (Qwen2.5 7B, lo stesso di Pm projexa sulla VM)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct-bnb-4bit', max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)

## 4. Addestramento

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
testi = [tokenizer.apply_chat_template(e['messages'], tokenize=False) for e in esempi]
dati = Dataset.from_dict({'text': testi}).shuffle(seed=42)
epoche = 3 if len(esempi) < 300 else 2
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dati,
    args=SFTConfig(dataset_text_field='text', max_seq_length=2048, per_device_train_batch_size=2,
                   gradient_accumulation_steps=4, num_train_epochs=epoche, learning_rate=2e-4,
                   warmup_steps=5, logging_steps=5, optim='adamw_8bit', fp16=True, bf16=False,
                   output_dir='uscita', report_to='none', seed=42))
trainer.train()

## 5. Prova veloce

In [ ]:
FastLanguageModel.for_inference(model)
prova = [esempi[0]['messages'][0], {'role': 'user', 'content': 'TRASCRIZIONE (nuovo tratto della riunione):\n[10:02] Cliente: Il file delle causali ve lo mandiamo a breve.'}]
ids = tokenizer.apply_chat_template(prova, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=60, temperature=0.2)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 6. Adattatore in formato Ollama (.gguf) e download

In [ ]:
model.save_pretrained('adattatore')
tokenizer.save_pretrained('adattatore')
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python llama.cpp/convert_lora_to_gguf.py adattatore --base-model-id Qwen/Qwen2.5-7B-Instruct --outtype f16 --outfile pm-projexa-adapter.gguf
import os
print('Dimensione:', round(os.path.getsize('pm-projexa-adapter.gguf') / 1e6), 'MB')
files.download('pm-projexa-adapter.gguf')

## 7. Fatto

Carica `pm-projexa-adapter.gguf` in Projexa: **Analisi PM → Modello sulla VM → Carica e installa**.
Projexa fa il backup del modello attuale, installa il nuovo e lo prova; se qualcosa non va torna da solo al precedente.
Quando sei sicuro che il nuovo è migliore, dalla stessa pagina puoi eliminare il backup.